In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

# 1. Definición de la Ecuación de Hill
def modelo_hill(C, A_max, K, n):
    return (A_max * (C**n)) / ((K**n) + (C**n))

# 2. Datos simulados de concentración de Hg2+ (mg/L) y respuesta óptica (UA)
# Límite OMS en agua potable: 0.002 mg/L
concentracion_Hg = np.array([0, 0.0005, 0.001, 0.0015, 0.002, 0.003, 0.005, 0.008, 0.012, 0.020])

# Respuesta simulada con ruido realista
np.random.seed(42)
absorbancia_teorica = modelo_hill(concentracion_Hg, A_max=1.5, K=0.0025, n=1.4)
ruido = np.random.normal(0, 0.02, size=len(concentracion_Hg))
absorbancia_medida = absorbancia_teorica + ruido
absorbancia_medida = np.clip(absorbancia_medida, 0, None)

# 3. Ajuste del modelo de Hill y cálculo de R²
popt, _ = curve_fit(modelo_hill, concentracion_Hg, absorbancia_medida, p0=[1.5, 0.002, 1.0])
A_max_fit, K_fit, n_fit = popt

A_pred_hill = modelo_hill(concentracion_Hg, *popt)
r2_hill = r2_score(absorbancia_medida, A_pred_hill)

# Dominio continuo para suavizar la curva de la gráfica
C_continuo = np.linspace(0, 0.02, 200)
A_fit_continuo = modelo_hill(C_continuo, *popt)

# 4. Gráfica 1: Curva de calibración completa con R²
plt.figure(figsize=(7, 4.5))
plt.scatter(concentracion_Hg, absorbancia_medida, color='blue', label='Datos simulados (Hg²⁺)')
plt.plot(C_continuo, A_fit_continuo, color='red', label=f'Ajuste Hill ($R^2 = {r2_hill:.4f}$)')
plt.axvline(x=0.002, color='orange', linestyle='--', label='Límite OMS (0.002 mg/L)')
plt.title('Grafica 1. Curva de calibración simulada para el biosensor de Hg²⁺')
plt.xlabel('Concentración de Hg²⁺ (mg/L)')
plt.ylabel('Absorbancia / Fluorescencia (UA)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 5. Rango Lineal (0 a 0.003 mg/L) y cálculo de R²
C_lineal = concentracion_Hg[concentracion_Hg <= 0.003]
A_lineal = absorbancia_medida[:len(C_lineal)]

m, b = np.polyfit(C_lineal, A_lineal, 1)
A_pred_lineal = m * C_lineal + b
r2_lineal = r2_score(A_lineal, A_pred_lineal)

# Cálculo de Límites de Detección (LOD) y Cuantificación (LOQ)
std_blanco = np.std(ruido)
LOD = (3.3 * std_blanco) / m
LOQ = (10 * std_blanco) / m

# 6. Gráfica 2: Rango lineal con R²
plt.figure(figsize=(7, 4.5))
plt.scatter(C_lineal, A_lineal, color='blue', label='Datos Rango Lineal')
plt.plot(C_lineal, A_pred_lineal, color='green', linestyle='--', 
         label=f'Ajuste Lineal ($R^2 = {r2_lineal:.4f}$)\ny = {m:.1f}x + {b:.3f}')
plt.axvline(x=0.002, color='orange', linestyle='--', label='Límite OMS')
plt.axhline(y=(m*LOD + b), color='purple', linestyle=':', label=f'LOD = {LOD:.5f} mg/L')
plt.title('Grafica 2. Rango lineal de trabajo para monitoreo de agua potable')
plt.xlabel('Concentración de Hg²⁺ (mg/L)')
plt.ylabel('Absorbancia / Fluorescencia (UA)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 7. Resumen de parámetros para la Tabla de Desempeño del Informe
print("=== PARÁMETROS DE DESEMPEÑO DEL BIOSENSOR ===")
print(f"A_max (ajustado): {A_max_fit:.4f} UA")
print(f"K (constante de afinidad): {K_fit:.6f} mg/L")
print(f"n (coeficiente de Hill): {n_fit:.4f}")
print(f"R² (Modelo completo de Hill): {r2_hill:.4f}")
print(f"Sensibilidad (m): {m:.2f} UA/(mg/L)")
print(f"R² (Rango lineal 0-0.003 mg/L): {r2_lineal:.4f}")
print(f"Límite de Detección (LOD): {LOD:.6f} mg/L")
print(f"Límite de Cuantificación (LOQ): {LOQ:.6f} mg/L")
